In [1]:
import torch
import CIFAR10.config as c
from utils import trainer as t

In [2]:
# CIFAR-10 Experiment 2: compare four RFA forward initializations with identical B matrices.
activation = torch.nn.Tanh()
hidden_layers = [1024, 512, 256, 128]
feedback_ranges = [0.025, 0.5, 0.1]
random_seeds = [147, 4562, 25823, 769283, 100000]
num_epochs = 50
checkpoint_interval = num_epochs + 1


def rfa_layers(net):
    return [layer for layer in net.modules() if hasattr(layer, "B")]


def match_balanced_weight_norms(balanced_net, xavier_net):
    """Scale each balanced layer to the corresponding Xavier Frobenius norm."""
    with torch.no_grad():
        for balanced_layer, xavier_layer in zip(
            rfa_layers(balanced_net), rfa_layers(xavier_net)
        ):
            balanced_norm = torch.linalg.vector_norm(balanced_layer.weight)
            xavier_norm = torch.linalg.vector_norm(xavier_layer.weight)
            if balanced_norm == 0:
                raise ValueError("Cannot norm-match a zero balanced weight matrix")
            balanced_layer.weight.mul_(xavier_norm / balanced_norm)


def copy_and_check_feedback_matrices(reference_net, networks):
    reference_layers = rfa_layers(reference_net)
    with torch.no_grad():
        for net in networks:
            for reference_layer, layer in zip(reference_layers, rfa_layers(net)):
                layer.B.copy_(reference_layer.B)

    all_equal = True
    for net in networks:
        for layer_index, (reference_layer, layer) in enumerate(
            zip(reference_layers, rfa_layers(net))
        ):
            equal = torch.equal(reference_layer.B, layer.B)
            print(f"B layer {layer_index}: {equal}")
            all_equal = all_equal and equal
    if not all_equal:
        raise RuntimeError("B matrices are not exactly equal; training was not started.")
    print("All B matrices are exactly equal before training.")


for random_seed in random_seeds:
    for feedback_range in feedback_ranges:
        run_prefix = f"rfa_{random_seed}_B{feedback_range:.3f}"
        configs = {}
        common_config = dict(
            activation=activation,
            hidden_layers=hidden_layers,
            SEED=random_seed,
            mode="RFA",
            feedback_range=feedback_range,
        )

        configs["balanced"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced",
            run_id=f"{run_prefix}_balanced",
            init_method="arora_balanced",
            arora_std=0.30,
        )
        configs["balanced_adversary"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced_1000",
            run_id=f"{run_prefix}_balanced_1000",
            init_method="arora_balanced",
            arora_std=1000.0,
        )
        configs["xavier"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_xavier",
            run_id=f"{run_prefix}_xavier",
            init_method="xavier",
        )
        configs["balanced_xavier_norm_matched"] = c.get_config(
            **common_config,
            run_name=f"{run_prefix}_balanced_xavier_norm_matched",
            run_id=f"{run_prefix}_balanced_xavier_norm_matched",
            init_method="arora_balanced",
            arora_std=0.30,
        )

        # Match only the dedicated balanced model; configs["balanced"] remains unchanged.
        match_balanced_weight_norms(
            configs["balanced_xavier_norm_matched"]["net"],
            configs["xavier"]["net"],
        )
        copy_and_check_feedback_matrices(
            configs["balanced"]["net"],
            [config["net"] for config in configs.values()],
        )

        for condition, config in configs.items():
            print(f"\nTraining {condition}: {run_prefix}")
            t.train_network(
                config,
                num_epochs=num_epochs,
                checkpoint_interval=checkpoint_interval,
            )


[LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA()]
[LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA()]
[LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA()]
Initializing layer LinearRFA() with method 'xavier'
Layer shape: torch.Size([1024, 3072]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer LinearRFA() with method 'xavier'
Layer shape: torch.Size([512, 1024]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer LinearRFA() with method 'xavier'
Layer shape: torch.Size([256, 512]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer LinearRFA() with method 'xavier'
Layer shape: torch.Size([128, 256]), nonlinearity: tanh, gain: 1.6666666666666667
Initializing layer LinearRFA() with method 'xavier'
Layer shape: torch.Size([10, 128]), nonlinearity: tanh, gain: 1.6666666666666667
[LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA(), LinearRFA()]
B layer 0: True
B layer 1: True
B layer 2: True
B layer 3: True


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Initializing Wandb:


wandb: Currently logged in as: g-aravindadithya (ICLR_2027) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


EPOCH:  1
Time:  2.986302137374878
EPOCH:  2
Time:  2.9963204860687256
EPOCH:  3
Time:  3.1365458965301514
EPOCH:  4
Time:  2.8970868587493896
EPOCH:  5
Time:  3.174065351486206
EPOCH:  6
Time:  4.546986103057861
EPOCH:  7
Time:  4.714199066162109
EPOCH:  8
Time:  4.641029119491577
EPOCH:  9
Time:  4.792357683181763
EPOCH:  10
Time:  6.484021902084351
EPOCH:  11
Time:  5.66739821434021
EPOCH:  12
Time:  4.617686748504639
EPOCH:  13
Time:  4.708310127258301
EPOCH:  14
Time:  4.693134307861328
EPOCH:  15
Time:  4.783961296081543
EPOCH:  16
Time:  4.526309490203857
EPOCH:  17
Time:  4.444601058959961
EPOCH:  18
Time:  4.709320545196533
EPOCH:  19
Time:  4.690263032913208
EPOCH:  20
Time:  6.451545476913452
EPOCH:  21
Time:  5.7888195514678955
EPOCH:  22
Time:  4.68707799911499
EPOCH:  23
Time:  4.601978063583374
EPOCH:  24
Time:  4.252784013748169
EPOCH:  25
Time:  5.770889043807983
EPOCH:  26
Time:  5.028459310531616
EPOCH:  27
Time:  5.498257875442505
EPOCH:  28
Time:  6.246150732040405

...

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


balancedness/B0,▁▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████
balancedness/B1,▁██▇▇▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇
balancedness/B2,▁██▇▇▇▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
balancedness/B3,▁███████████████████████████████████████
balancedness/||W0W0T||_F,▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
balancedness/||W0W0^T - W1^TW1||_F,▁▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████
balancedness/||W1TW1||_F,▁▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████
balancedness/||W1W1T||_F,▁▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████
balancedness/||W1W1^T - W2^TW2||_F,▁██▇▇▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇
balancedness/||W2TW2||_F,▁██▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
+12,...


FINISHED TRAINING :)

Training xavier: rfa_147_B0.025
Initializing Wandb:


EPOCH:  1
Time:  6.181497812271118
EPOCH:  2
Time:  4.9012064933776855
EPOCH:  3
Time:  4.136836290359497
EPOCH:  4
Time:  4.833017110824585
EPOCH:  5
Time:  4.821996450424194
EPOCH:  6
Time:  5.568019151687622
EPOCH:  7
Time:  6.761781454086304
EPOCH:  8


: 

# Experiment 2
RFA feedback-scale sensitivity sweep: Arora-balanced versus Xavier initialization.

Feedback matrices use $B_{ij} \sim \mathrm{Uniform}(-b,b)$ with $b \in \{0.025, 0.05, 0.10\}$. CIFAR-10 runs use 50 epochs.